# Notebook 19: Hospital Readmissions in Michigan (CMS Hospital Readmissions Reduction Program)

**Question:** For which conditions do Michigan hospitals have more readmissions than expected, and what does that suggest about post-acute handoffs?

**Data:** CMS Provider Data Catalog, Hospital Readmissions Reduction Program results, pulled through the CMS public API.

**New skills:** API access with paging; handling "Not Available" and "Too Few to Report" values; reshaping long to wide.

**Rules:** Own numbers only. Definitions from CMS documentation. Hospital naming decision to be made before results are viewed.

In [0]:
import requests, time
import pandas as pd

DATASET = "9n3s-kdb3"
URL = f"https://data.cms.gov/provider-data/api/1/datastore/query/{DATASET}/0"
LIMIT = 500
TRANSIENT = {429, 500, 502, 503, 504}

def get_page(offset, max_tries=5):
    params = {"limit": LIMIT, "offset": offset, "results": "true"}
    if offset == 0:
        params["count"] = "true"
    for attempt in range(1, max_tries + 1):
        try:
            r = requests.get(URL, params=params, timeout=60)
            if r.status_code in TRANSIENT:
                raise requests.ConnectionError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r.json()
        except (requests.ConnectionError, requests.Timeout) as e:
            if attempt == max_tries:
                raise
            wait = 2 ** attempt
            print(f"  offset {offset}: attempt {attempt} failed ({e}), retrying in {wait}s")
            time.sleep(wait)

first = get_page(0)
total = int(first["count"])
rows = list(first["results"])
print(f"API reports {total:,} rows")

offset = 0
while len(rows) < total:
    offset += LIMIT
    batch = get_page(offset)["results"]
    if not batch:
        break
    rows.extend(batch)
    if (offset // LIMIT) % 10 == 0:
        print(f"  {len(rows):,} of {total:,} rows")
    time.sleep(0.3)

print(f"Rows collected: {len(rows):,}")
assert len(rows) == total, f"Collected {len(rows):,} rows, but the API reports {total:,}"

pd.DataFrame(rows).astype(str).pipe(spark.createDataFrame) \
  .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.hrrp_fy2026_bronze")
print("Saved to hrrp_fy2026_bronze")

**Cross-check attempted:** the bulk CSV download returned HTTP 403 (Forbidden) from code, so the API-vs-file comparison was not run. The API load passed its completeness check (18,330 of 18,330 rows).

In [0]:
from pyspark.sql import functions as F

b = spark.table("workspace.default.hrrp_fy2026_bronze")
MISSING = ["N/A", "Too Few to Report", "", "nan", "None"]

def num(c):
    return F.when(F.col(c).isin(MISSING), None).otherwise(F.col(c)).cast("double")

COND = {"READM-30-AMI-HRRP": "Heart attack", "READM-30-CABG-HRRP": "Bypass surgery",
        "READM-30-COPD-HRRP": "COPD", "READM-30-HF-HRRP": "Heart failure",
        "READM-30-HIP-KNEE-HRRP": "Hip/knee replacement", "READM-30-PN-HRRP": "Pneumonia"}
cmap = F.create_map(*[F.lit(x) for kv in COND.items() for x in kv])

silver = (b.withColumn("condition", cmap[F.col("measure_name")])
           .withColumn("err", num("excess_readmission_ratio"))
           .withColumn("discharges", num("number_of_discharges"))
           .withColumn("readmissions", num("number_of_readmissions"))
           .withColumn("predicted_rate", num("predicted_readmission_rate"))
           .withColumn("expected_rate", num("expected_readmission_rate")))

# Guardrail 1: every measure must map to a known condition
unmapped = silver.filter(F.col("condition").isNull()).count()
assert unmapped == 0, f"{unmapped:,} rows have an unrecognized measure name"

# Guardrail 2: no value may fail conversion unless it's a known 'missing' marker
bad = silver.filter(F.col("excess_readmission_ratio").isNotNull()
                    & ~F.col("excess_readmission_ratio").isin(MISSING) & F.col("err").isNull()).count()
assert bad == 0, f"{bad:,} ratio values failed to convert"

silver.select("facility_id", "state", "condition", "err", "discharges", "readmissions",
              "predicted_rate", "expected_rate", "footnote", "start_date", "end_date") \
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.hrrp_fy2026_silver")

s = spark.table("workspace.default.hrrp_fy2026_silver")
print(f"Silver rows: {s.count():,} | with a ratio: {s.filter(F.col('err').isNotNull()).count():,}")
display(s.groupBy("footnote").agg(F.count("*").alias("rows"),
                                  F.sum(F.when(F.col("err").isNull(), 1).otherwise(0)).alias("rows_missing_ratio"))
         .orderBy(F.desc("rows")))

In [0]:
import requests

meta = requests.get("https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items", timeout=60).json()
fn = [(d.get("identifier"), d.get("title")) for d in meta if "footnote" in d.get("title", "").lower()]
print("Footnote datasets found:", fn)

if fn:
    rid = fn[0][0]
    res = requests.get(f"https://data.cms.gov/provider-data/api/1/datastore/query/{rid}/0",
                       params={"limit": 500}, timeout=60).json()["results"]
    print(f"\n{len(res)} footnote definitions:")
    for row in res:
        print(" | ".join(str(v) for v in row.values()))

In [0]:
s = spark.table("workspace.default.hrrp_fy2026_silver")
rated = s.filter(F.col("err").isNotNull())

def summarize(df, label):
    return (df.groupBy("condition")
              .agg(F.count("*").alias("hospitals"),
                   F.round(F.avg(F.when(F.col("err") > 1, 1).otherwise(0)) * 100, 1).alias("pct_above_expected"),
                   F.round(F.expr("percentile_approx(err, 0.5)"), 4).alias("median_err"))
              .withColumn("scope", F.lit(label)))

print(f"Michigan hospitals in the program: {s.filter(F.col('state') == 'MI').select('facility_id').distinct().count()}")
res = summarize(rated.filter(F.col("state") == "MI"), "Michigan").unionByName(summarize(rated, "National"))
display(res.orderBy("condition", "scope"))

print("Sensitivity check: Michigan without footnote-29 results")
display(summarize(rated.filter((F.col("state") == "MI") & (F.col("footnote") != "29")), "Michigan excl. 29")
        .orderBy("condition"))

In [0]:
hk = rated.filter(F.col("condition") == "Hip/knee replacement")
mi_hk = hk.filter(F.col("state") == "MI")
n = mi_hk.count()
above = mi_hk.filter("err > 1").count()
p_nat = hk.filter("err > 1").count() / hk.count()
print(f"Michigan hip/knee hospitals above expected: {above} of {n}")
print(f"If Michigan matched the nation ({p_nat:.1%}), we'd expect about {n * p_nat:.0f} of {n}")

display(mi_hk.agg(F.round(F.avg("predicted_rate"), 2).alias("avg_predicted_rate_pct"),
                  F.round(F.avg("expected_rate"), 2).alias("avg_expected_rate_pct"),
                  F.round(F.expr("percentile_approx(err, 0.25)"), 3).alias("p25_err"),
                  F.round(F.expr("percentile_approx(err, 0.75)"), 3).alias("p75_err"),
                  F.sum(F.when(F.col("err") > 1.10, 1).otherwise(0)).alias("hospitals_over_1_10")))

## Findings: Notebook 19

**Data:** CMS Hospital Readmissions Reduction Program, FY 2026 results (performance period July 2021 to June 2024), 18,330 hospital-condition rows, pulled through the CMS public API.

**Pipeline notes:**
- The API returned a temporary 503 error mid-load; retry with exponential backoff recovered automatically, and a completeness check confirmed all 18,330 rows.
- The planned fallback (bulk CSV download) returned 403 Forbidden from code. A fallback that has never been tested is not a fallback.

**Missing results (36% of rows, never treated as zero):** footnote 1, too few cases (3,150); footnote 5, not available for the period (3,255); footnote 7, no qualifying cases (205). Definitions from the CMS Footnote Crosswalk.

**Partial-period results (footnote 29):** 377 results nationally (about 3%) were calculated with partial-period data under a CMS-approved exception. In Michigan, 44 results (about 12%) carry this footnote, roughly four times the national rate; the data does not say why. Results were tested with and without these rows.

**How to read the ratio:** The Excess Readmission Ratio compares each hospital with an average hospital treating similar patients, so it centers near 1.0 nationally by design. Above 1.0 means more readmissions than expected.

**Michigan compared with the nation (share of hospitals above expected):**
- **Hip/knee replacement: 75.6% in Michigan vs. 47.9% nationally.** 31 of 41 hospitals, where about 20 would be expected if Michigan matched the nation. Unchanged without partial-period results (75.0%).
- The pattern is broad: even the 25th-percentile Michigan hospital is above 1.0, and 14 of 41 are at least 10% above expected.
- In real rates: 6.38% predicted vs. 5.99% expected on average, about 0.4 percentage points higher. Consistent, but modest per patient.
- Michigan is better than the nation on COPD (36.4% vs. 47.2%) and bypass surgery (36.7% vs. 49.9%, 30 hospitals). Heart attack, heart failure, and pneumonia are close to national.

**What I'd tell a COO:** Three out of four Michigan hospitals have more hip and knee readmissions than expected, a consistent statewide pattern. The discharge pathway is the first place I'd investigate, because joint replacement is planned surgery and readmissions depend heavily on where patients go and the support they have at home. But I'd confirm the cause before funding a fix.

**What this data can't answer:** Why Michigan is higher; what is happening now (the data ends June 2024); whether the gap reflects care quality or patient differences not captured in the risk adjustment; and why Michigan has more partial-period exceptions.

**What I'd need next:** Readmission diagnoses, discharge destination, days from discharge to readmission, and the post-acute provider for joint replacement patients. It is the same data gap that limited the readmission analysis in Notebook 12.

**Publishing rule:** No hospital names in any output; results reported by condition and state only.